# Causal Impact of the 2025 F5 Cybersecurity Breach

F5 (FFIV) experienced a major cybersecurity breach in October 2025.  
This project estimates how much the breach impacted FFIV’s daily stock returns.

Because we cannot run experiments on real stock prices,  
we use observational causal inference (Difference-in-Differences + trend tests).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import statsmodels.formula.api as smf
import yfinance as yf

Load Market Data: Fetch daily adjusted prices for FFIV, its closest competitors, and market benchmarks from Yahoo Finance.

In [ ]:
# Pulls daily adjusted close prices for one ticker. Returns: date, price, ticker
def get_daily_prices(ticker, start_date, end_date):
    # Download historical data from Yahoo Finance
    df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    df = df.reset_index()
    df = df.rename(columns={"Date": "date"})

    # Use Adjusted Close if available, else use Close
    if "Adj Close" in df.columns:
        df["price"] = df["Adj Close"]
    else:
        # If Adjusted Close is not available, use Close price
        df["price"] = df["Close"]

    # Add ticker column
    df["ticker"] = ticker
    return df[["date", "price", "ticker"]]

# Downloads daily prices for a list of tickers and concatenates into one DataFrame
def get_prices(tickerlist, start, end):
    frames = []

    # Loop through each ticker and get daily prices
    for ticker in tickerlist:
        frames.append(get_daily_prices(ticker, start, end))

    # Combine all dataframes into one
    return pd.concat(frames, ignore_index=True)

# Runs Difference-in-Differences regression
# Estimates the causal effect of an event on stock returns
# FIX (Nov 2025): the original version filtered to date <= event_date, so the
# "post" period was only the event day itself. A real DiD needs a post window
# of real length, so the window now runs post_days AFTER the event too.
def run_diff_in_diff(df, event_date, treated, pre_days=180, post_days=30):
    start_date = event_date - datetime.timedelta(days=pre_days)
    end_date = event_date + datetime.timedelta(days=post_days)

    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]

    df = df.copy()

    # Treatment indicator: 1 if treated ticker, else 0
    df.loc[:, 'treated'] = np.where(df['ticker'] == treated, 1, 0)

    # Post-event indicator: 1 if date >= event_date, else 0
    df.loc[:, 'post'] = np.where(df['date'] >= event_date, 1, 0)

    # Interaction term: treated * post
    df.loc[:, 'treated_post'] = df['treated'] * df['post']

    # DID regression model
    # Returns ~ treated + post + treated_post
    formula = "returns ~ treated + post + treated_post"
    model = smf.ols(formula=formula, data=df).fit()

    return model

# Trends Analysis
# Estimates the causal effect of an event on stock returns using trends
def difference_in_trends (df, event_date, treated, pre_days=180, post_days=30):
    start_date = event_date - datetime.timedelta(days=pre_days)
    end_date = event_date + datetime.timedelta(days=post_days)

    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    df = df.copy()

    df['treated'] = np.where(df['ticker'] == treated, 1, 0)
    df['trend'] = (event_date - df['date']).dt.days
    df['treated_trend'] = df['treated'] * df['trend']

    # Trends regression model
    # Returns ~ treated + trend + treated_trend
    formula = "returns ~ treated + trend + treated_trend"
    model = smf.ols(formula=formula, data=df).fit()

    return model


In [ ]:
# Defining the analysis window of measure
start_date = '2025-04-18'
end_date   = '2025-12-12'

# Defining the breach announcement date
event_date = pd.to_datetime('2025-10-16')

#List fo tickers includedd in the analysis
tickerlist = [
    "FFIV",   # treated (F5 Networks)
    "AKAM",   # Akamai – closest competitor in CDN + WAF
    "NET",    # Cloudflare – modern WAF + load balancing + edge security
    "CSCO",   # Cisco – enterprise networking + load balancing/security

    # Market controls
    "SPY",    # S&P 500 ETF – broad market benchmark
    "VTI"     # Total U.S. stock market ETF – full market control
]

# Creating the prices dataframe
prices = get_prices(tickerlist, start_date, end_date)

# Flatten the columns (yfinance returns a MultiIndex) — FIX: this line used to
# sit BEFORE prices was created, so the notebook only ran out of order.
prices.columns = prices.columns.get_level_values(0)

prices = prices.sort_values(["ticker", "date"])

# Compute daily percentage returns for each stock
prices["returns"] = prices.groupby("ticker")["price"].pct_change()


In [ ]:
# Plotting the impact of the event
ffiv=prices[prices['ticker'] == 'FFIV']
akam=prices[prices['ticker'] == 'AKAM']
spy=prices[prices['ticker'] == 'SPY']
net=prices[prices['ticker'] == 'NET']
cisco=prices[prices['ticker'] == 'CSCO']

# Plot daily returns for each stock
plt.plot(ffiv['date'], ffiv['returns'], label='FFIV')
plt.plot(akam['date'], akam['returns'], label='AKAM')
plt.plot(net['date'],  net['returns'], label='NET')
plt.plot(spy['date'],  spy['returns'], label='SPY')
plt.plot(cisco['date'], cisco['returns'], label='CSCO')
plt.axvline(event_date, color='red', linestyle='--')

# Add titles and labels
plt.title('Daily Stock Returns: F5 vs Competitors Around Event Date')
plt.xlabel('Date')
plt.ylabel('Daily Returns(%)')

# Add legend and show plot
plt.legend()

# Display the plot
plt.show()


In [ ]:
# Display first 10 rows of FFIV data after event date
ffiv_after = prices[(prices['ticker'] == 'FFIV') & (prices['date'] >= event_date)]
ffiv_after.head(10)


## Methods

### Difference-in-Differences (DiD)
Estimates stock return impact after the breach by comparing:
- FFIV (treated)
- vs competitors (controls)

Formula:
returns ~ treated + post + treated_post

The coefficient on `treated_post` = estimated causal effect.

In [ ]:
# Running Difference-in-Differences analysis
event_date = pd.to_datetime('2025-10-16')
treated = 'FFIV'

# Headline result: 30 days after the breach
model = run_diff_in_diff(prices, event_date, treated, post_days=30)
print("=== 30-day post window ===")
print(model.summary())

# Fade check: does the effect last 90 days?
model_90 = run_diff_in_diff(prices, event_date, treated, post_days=90)
print("=== 90-day post window ===")
print(model_90.summary())


## Assumptions

### Parallel Trend Assumption
The Parallel Trend Assumption requires that in the absence of treatment, the difference between the ‘treatment’ and ‘control’ group is constant over time.

In [ ]:
# Parallel Trends Assumption Check
# Plot pre-event daily returns for all stocks
pre_event = prices[prices['date'] < event_date]

plt.figure(figsize=(12,6))

# Loop through each ticker and plot pre-event returns
# FIX: only plot tickers actually downloaded above (FTNT, CHKP, CRWD, ZS
# were listed here before but never fetched, so their lines were empty).
for t in ["FFIV", "AKAM", "NET", "CSCO", "SPY", "VTI"]:
    se = pre_event[pre_event['ticker']==t]
    plt.plot(se['date'], se['returns'], alpha=0.4)

# Add titles and labels
plt.title("Pre-Event Daily Returns (Parallel Trend Assumption Check)")
plt.axvline(event_date, color='red', linestyle='--') # Event date line
plt.grid(True)
plt.show()


### C. Placebo Test
We fake the event a month earlier (Sep 16, 2025) to check if there's a false signal.
The fake date must sit far enough back that its whole post window ends BEFORE
the real event — a fake date 7 days before the breach would swallow the real
breach in its "post" period and look significant for the wrong reason.


In [ ]:
# Placebo Test: Run Difference-in-Differences analysis with a placebo event date
# Fake event Sep 16, 2025 — 30 days before the real breach — with a 29-day post
# window so the placebo window ends Oct 15, the day BEFORE the real event.
placebo_date = pd.to_datetime('2025-09-16')

# Run Difference-in-Differences analysis with placebo event date
model_placebo = run_diff_in_diff(prices, placebo_date, "FFIV", post_days=29)
print(model_placebo.summary())


Results show that in the pre preiod we have no evidence that theres any differnece in trend and so no reason to believe that it would change afterwards.

### D. Trend-Based Robustness Test
Controls for pre-event slope differences.

In [ ]:
#Checking that FFIV and the control tickers had *parallel trends* prior 
# to the breach
test_trend_model = difference_in_trends(prices, event_date, "FFIV")
print(test_trend_model.summary())